# 07. Segmentation Test

- Goal: inspect rep/phase segmentation readiness and frame-level phase outputs.
- Docs: `docs_eng/pipeline/07_segmentation.md` / `docs/pipeline/07_segmentation.md`
- Inputs: Annotated and normalized pose dataframe from prior-stage cells.
- Outputs: In-memory dataframes/reports unless a cell explicitly saves under `data/processed/`.
- Checks: Rep/phase labels, not-assessed cases, and segmentation provenance summaries.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json

import pandas as pd

from movement.annotation import apply_annotation, load_annotation_csv
from movement.config import LANDMARKS
from movement.exercise_definition import load_exercise_definition
from movement.io import load_pose_csv
from movement.normalization import normalize_pose_by_hip_torso
from movement.pipeline import (
    AnnotationConfig,
    ExerciseDefinitionConfig,
    NormalizationConfig,
    PhaseSegmentationConfig,
    PipelineConfig,
    ValidationConfig,
    run_pipeline,
)
from movement.segmentation import PhaseSegmentationReport, segment_phases
from movement.validation import run_basic_validation
from movement.config import make_coordinate_columns, make_required_columns, make_visibility_columns

print('imports OK')

## Data Setup

Mirrors required upstream steps ①–⑤ to produce a normalized dataframe with annotation and exercise definition; the optional floor-relative normalization filter is omitted here.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing pyproject.toml")
    PROJECT_ROOT = PROJECT_ROOT.parent

csv_path = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv"
ann_path = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_annotation.csv"
def_dir  = PROJECT_ROOT / "data/definitions/exercises"

df_raw = load_pose_csv(csv_path)

run_basic_validation(
    df=df_raw,
    required_columns=make_required_columns(LANDMARKS),
    coordinate_columns=make_coordinate_columns(LANDMARKS),
    visibility_columns=make_visibility_columns(LANDMARKS),
)

ann_df = load_annotation_csv(ann_path)
df_ann, _ = apply_annotation(df_raw, ann_df)

exercise_def = load_exercise_definition(exercise_id='squat', definitions_dir=def_dir)
assert exercise_def.phase_segmentation is not None, 'squat.yaml must have phase_segmentation block'

df_norm, _ = normalize_pose_by_hip_torso(df=df_ann, landmarks=LANDMARKS)

print(f'normalized: {df_norm.shape[0]} frames')
print(f'rep frames: {(df_norm["segment_type"] == "rep").sum()}')
print(f'phase_segmentation spec: {exercise_def.phase_segmentation}')

## Recording-Specific Phase Guideline

`p01_squat_set1_phase_split.csv`, when present, is a visual/QC guide for this recording only. It is not ground truth phase annotation and is not used as pipeline scoring input.

In [ ]:
phase_guideline_path = ann_path.with_name("p01_squat_set1_phase_split.csv")
phase_guideline_df = pd.DataFrame()

if phase_guideline_path.exists():
    phase_guideline_df = pd.read_csv(phase_guideline_path)
    print("phase guideline:", phase_guideline_path.relative_to(PROJECT_ROOT))
    print("policy: guide only; not confirmed annotation; not scoring input")
    print("rows:", len(phase_guideline_df))
    print("reps:", sorted(phase_guideline_df["rep_id"].dropna().astype(int).unique().tolist()))

    rep_ranges = (
        ann_df.loc[ann_df["segment_type"].eq("rep"), ["set_id", "rep_id", "start_frame", "end_frame"]]
        .rename(columns={"start_frame": "annotation_start", "end_frame": "annotation_end"})
        .copy()
    )
    guide_ranges = (
        phase_guideline_df.groupby(["set_id", "rep_id"], dropna=False)
        .agg(
            guideline_start=("start_frame", "min"),
            guideline_end=("end_frame", "max"),
            guideline_phases=("phase", lambda s: " / ".join(s.astype(str))),
        )
        .reset_index()
    )
    coverage = rep_ranges.merge(guide_ranges, on=["set_id", "rep_id"], how="left")
    coverage["covers_annotation_rep"] = (
        coverage["annotation_start"].eq(coverage["guideline_start"])
        & coverage["annotation_end"].eq(coverage["guideline_end"])
    )
    display(coverage)
    display(phase_guideline_df.head(12))
else:
    print("No phase guideline CSV found beside the annotation file.")
    print("Pipeline phase segmentation will be inspected without recording-specific phase guidance.")


## Direct segment_phases() Test

In [ ]:
df_seg, reports = segment_phases(df_norm, exercise_def, fps_default=30.0)

print(f'output shape: {df_seg.shape}')
print(f'PhaseSegmentationReport count: {len(reports)}')
for r in reports:
    print(f'  rep {r.rep_id}: inflection_frames={r.inflection_frames}  phase_assignments={r.phase_assignments}')

## Check 1: `phase` column populated for rep frames

In [ ]:
assert 'phase' in df_seg.columns, 'phase column missing'

rep_mask = df_seg['segment_type'] == 'rep'
nrep = int(rep_mask.sum())
n_labeled = int(df_seg.loc[rep_mask, 'phase'].notna().sum())
print(f'rep frames: {nrep}  labeled: {n_labeled}')
if n_labeled < nrep:
    print('NOTE: p01 phase labels are incomplete; inspect PhaseSegmentationReport rejection reasons below.')


## Check 2: Kinematic labels are Descent / Ascent / Bottom_Hold

In [ ]:
valid_labels = {'Descent', 'Ascent', 'Bottom_Hold'}
actual = set(df_seg.loc[rep_mask, 'phase'].dropna().unique())
print(f'labels found: {actual}')
assert actual.issubset(valid_labels), f'unexpected labels: {actual - valid_labels}'
if {'Descent', 'Ascent'}.issubset(actual):
    print('PASS: Descent and Ascent labels are present')
else:
    print('NOTE: Descent/Ascent labels are not both present for this p01 run.')

counts = df_seg.loc[rep_mask, 'phase'].value_counts()
for label, cnt in counts.items():
    print(f'  {label}: {cnt} frames')


## Check 3: Non-rep frames remain NA

In [ ]:
non_rep_mask  = df_seg['segment_type'] != 'rep'
n_non_rep     = non_rep_mask.sum()
n_non_rep_na  = df_seg.loc[non_rep_mask, 'phase'].isna().sum()
print(f'non-rep frames: {n_non_rep}  still NA: {n_non_rep_na}')
assert n_non_rep_na == n_non_rep, 'non-rep frames must have NA phase'
print('PASS: non-rep frames remain NA')

## Check 4: PhaseSegmentationReport structure

In [ ]:
for r in reports:
    assert isinstance(r, PhaseSegmentationReport)
    assert isinstance(r.rep_id, int)
    assert isinstance(r.inflection_frames, list)
    assert isinstance(r.phase_assignments, dict)
    d = r.as_dict()
    assert 'rep_id' in d
    assert 'inflection_frames' in d
    assert 'phase_assignments' in d
    assert 'smoothing_method' in d

report_df = pd.DataFrame([r.as_dict() for r in reports])
display(report_df[['rep_id', 'rejected_reason', 'inflection_frames', 'multi_inflection_collapsed']])
print(f'PASS: PhaseSegmentationReport structure valid for all {len(reports)} reps')


## Check 5: Inflection count matches multi_inflection_policy=global_extremum

In [ ]:
inflection_counts = [len(r.inflection_frames) for r in reports]
summary = pd.Series(inflection_counts, name='num_inflections').value_counts().sort_index()
display(summary.to_frame())
print('NOTE: global_extremum should produce one inflection when the reference trajectory is usable; zero means not assessed for that rep.')


## Check 6: Pipeline Integration

In [ ]:
cfg = PipelineConfig()
cfg.validation = ValidationConfig(enabled=True)
cfg.annotation = AnnotationConfig(enabled=True, path=ann_path)
cfg.exercise_definition = ExerciseDefinitionConfig(
    enabled=True,
    definitions_dir=def_dir,
    exercise_id='squat',
)
cfg.normalization = NormalizationConfig(enabled=True)
cfg.phase_segmentation = PhaseSegmentationConfig(enabled=True, fps_default=30.0)

pipe_df, pipe_report = run_pipeline(
    df_raw,
    config=cfg,
    landmarks=LANDMARKS,
    ann_df=ann_df,
)

assert 'phase_segmentation' in pipe_report, 'phase_segmentation key missing in report'
ps_report = pipe_report['phase_segmentation']
assert len(ps_report) >= 1, 'no reps processed'

rep_mask_pipe = pipe_df['segment_type'] == 'rep'
n_labeled_pipe = int(pipe_df.loc[rep_mask_pipe, 'phase'].notna().sum())
print(f'pipeline ⑦ phase_segmentation report: {len(ps_report)} reps')
print(f'phase column labeled for {n_labeled_pipe} / {int(rep_mask_pipe.sum())} rep frames')
print(f'steps executed: {list(pipe_report.keys())}')


## Check Summary

This notebook cell is a compact execution/QC checkpoint. Use the pipeline document linked in the header for definitions, interpretation policy, and scope.
